# Bandit Regret Analysis

Reproduces paper Figure 3: cumulative regret curves for UCB1, Thompson,
and Oracle over T=10,000 rounds with K=3 arms.

Outputs `paper/figures/bandit_regret.svg`.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from flashspec.bandit import UCB1Selector, ThompsonSelector, OracleSelector
from flashspec.utils.device import set_seed
print('Imports OK')

In [ ]:
# ── Experiment parameters ─────────────────────────────────────────────────────
T = 10_000
TRUE_RATES = [0.30, 0.70, 0.50]   # arm 1 is optimal (Δ=0.4)
K = len(TRUE_RATES)
BEST_RATE = max(TRUE_RATES)
N_RUNS = 5  # average over this many seeds
print(f'K={K}, T={T}, true_rates={TRUE_RATES}, best={BEST_RATE}')

In [ ]:
def run_bandit(selector, true_rates, T, seed):
    """Run one bandit for T rounds; return per-round cumulative regret array."""
    rng = np.random.default_rng(seed)
    best = max(true_rates)
    cum_regret = np.zeros(T)
    for t in range(T):
        arm = selector.select()
        accepted = int(rng.random() < true_rates[arm])
        selector.update(arm, accepted)
        cum_regret[t] = (cum_regret[t-1] if t > 0 else 0) + (best - true_rates[arm])
    return cum_regret

results = {}
for name, cls, kwargs in [
    ('UCB1',     UCB1Selector,     dict(n_arms=K, window_size=0)),
    ('Thompson', ThompsonSelector, dict(n_arms=K)),
    ('Oracle',   OracleSelector,   dict(n_arms=K, true_rates=TRUE_RATES)),
]:
    runs = []
    for seed in range(N_RUNS):
        set_seed(seed)
        selector = cls(**kwargs)
        runs.append(run_bandit(selector, TRUE_RATES, T, seed))
    results[name] = np.array(runs)  # (N_RUNS, T)
    mean_final = results[name][:, -1].mean()
    print(f'{name}: mean cumulative regret at T={T}: {mean_final:.1f}')

In [ ]:
# ── Plot and save to paper/figures/ ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ts = np.arange(1, T + 1)
colors = {'UCB1': '#2563eb', 'Thompson': '#16a34a', 'Oracle': '#dc2626'}

for name, data in results.items():
    mean_r = data.mean(axis=0)
    std_r  = data.std(axis=0)
    ax.plot(ts, mean_r, label=name, color=colors[name], linewidth=2)
    ax.fill_between(ts, mean_r - std_r, mean_r + std_r,
                    color=colors[name], alpha=0.15)

# Theoretical bound: c * sqrt(K * T * log(T))
import math
bound = np.array([math.sqrt(K * t * math.log(max(t, 2))) for t in ts])
ax.plot(ts, bound, 'k--', linewidth=1, label=r'$O(\sqrt{KT\log T})$')

ax.set_xlabel('Round $t$', fontsize=12)
ax.set_ylabel('Cumulative Regret $R_t$', fontsize=12)
ax.set_title(f'Bandit Regret: K={K}, true rates={TRUE_RATES}', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()

out_dir = Path('../paper/figures')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'bandit_regret.svg'
fig.savefig(out_path, format='svg')
plt.close()
print(f'Saved: {out_path}')